In [1]:
import sys
sys.path.append('../') 
import DirectStiffness as DS

In [2]:
DS.Structure()

Structure init called


In [3]:
cfg = DS.Config()

2026-05-04 20:33:50.181 | DEBUG    | DirectStiffness.utils:__init__:10 - Config init called
2026-05-04 20:33:50.302 | DEBUG    | DirectStiffness.utils:_select_device:35 - NVIDIA GPU detected : NVIDIA GeForce RTX 3060 Laptop GPU
2026-05-04 20:33:50.302 | DEBUG    | DirectStiffness.utils:__init__:19 - Using cuda as the default device.


In [2]:
import numpy as np

In [ ]:
E_M=[2*10**8,2*10^8,2*10^8,2*10^8,2*10^8]                             # modulus of Elasticity matrix 
dep=0.1
base=.05
I_M=[1*10^(-5),1*10^(-5),1*10^(-5),1*10^(-5),1*10^(-5)]    # moment of inertia matrix
A_M=[0.01,0.01,0.01,0.01,0.01]    # cross-section area matrix

In [32]:
node=[[0, 0],[0, 3],[3, 3],[3, 0]] # nodal coordinates
conn=[[0, 1],[1, 2],[2, 3]]         #element connection matrix       
NN= len(node)        #number of nodes
NE= len(conn)        #number of elements
ndof= 3*NN              #total dof
fdof=[4, 5, 6, 7, 8, 9, 12]         #free dof

In [33]:
len(node[0])

2

In [34]:
conn[1][0]

1

In [35]:
# Formation of Displacement Matrix
d=np.zeros([ndof,1])                                                            
f=np.zeros([ndof,1])       
f[3]=10
#f(5)=-10;f(11)=-10;%empty load vector
#f(2)=-4;f(3)=-2.66666666666667;f(5)=-4-2-10;f(6)=+2.66666666666667-0.666666666666667;f(8)=-2;f(9)=0.666666666666667;

In [36]:
f, conn

(array([[ 0.],
        [ 0.],
        [ 0.],
        [10.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.]]),
 [[0, 1], [1, 2], [2, 3]])

In [48]:
# Formation of Stiffness Matrix
K= np.zeros(ndof)          #size of assembled stiffness matrix is ndof x ndof
for e in range (0,NE):              #loop over all elements
    print("----------------------")
    print("elem",e)
    n1=conn[e][0]       #ID of first node of each element
    print("n1",n1)
    n2=conn[e][1]       #ID of second node of each element
    print("n2",n2)

    x1=node[n1][0]      #x coordinate of node n1
    print("x1",x1)
    x2=node[n2][0]      #x coordinate of node n2
    print("x2",x2)
    y1=node[n1][1]      #y coordinate of node n1
    print("y1",y1)
    y2=node[n2][1]      #y coordinate of node n2
    print("y2",y2)

    L=((x2-x1)**2+(y2-y1)**2)**0.5 # perform sqrt using **0.5 ## sqrt((x2-x1)^2+(y2-y1)^2);
    print("L elem", L)
    theta=np.arctan2((y2-y1),(x2-x1))
    print("theta", np.degrees(theta))
    c=np.cos(theta)
    s=np.sin(theta)
    E=E_M[e]
    I=I_M[e]
    A=A_M[e]
    K_W_R=  k = np.array([
    [E*A/L,      0,            0,           -E*A/L,      0,            0],
    [0,      12*E*I/L**3,   6*E*I/L**2,     0,      -12*E*I/L**3,   6*E*I/L**2],
    [0,       6*E*I/L**2,   4*E*I/L,        0,      -6*E*I/L**2,    2*E*I/L],
    [-E*A/L,     0,            0,            E*A/L,      0,            0],
    [0,     -12*E*I/L**3,  -6*E*I/L**2,    0,       12*E*I/L**3,  -6*E*I/L**2],
    [0,       6*E*I/L**2,   2*E*I/L,        0,      -6*E*I/L**2,    4*E*I/L]
    ])
    if (y2-y1)==0: # if not inclined
        KE=K_W_R
    else:
        R=np.array([[c, s, 0, 0, 0, 0],
                    [-s, c, 0, 0, 0, 0],
                    [ 0, 0, 1, 0, 0, 0],
                    [ 0, 0, 0, c, s, 0],
                    [ 0, 0, 0, -s, c, 0],
                    [ 0, 0, 0, 0, 0, 1]])
    KE=np.matmul(np.transpose(R),(K_W_R),R)  

----------------------
elem 0
n1 0
n2 1
x1 0
x2 0
y1 0
y2 3
L elem 3.0
theta 90.0
----------------------
elem 1
n1 1
n2 2
x1 0
x2 3
y1 3
y2 3
L elem 3.0
theta 0.0
----------------------
elem 2
n1 2
n2 3
x1 3
x2 3
y1 3
y2 0
L elem 3.0
theta -90.0


TODO: Assembly logic adn then carry on
https://github.com/praksharma/Programming/blob/main/FEM/thesis/myGUI/analysis_1.m